# 🥦 Nexora — Product Intelligence & Association EDA
**Phase 4: Multi-Level EDA | Notebook 03**

### Objective
Examine bestselling products, department velocity, reorder stickiness, add-to-cart position dynamics, and itemset co-occurrence associations across catalog products.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_DIR = Path("../data/processed")
df_products = pd.read_csv(DATA_DIR / "products.csv")
df_depts = pd.read_csv(DATA_DIR / "departments.csv")
df_aisles = pd.read_csv(DATA_DIR / "aisles.csv")
df_op_sample = pd.read_csv(DATA_DIR / "order_products.csv", nrows=4000000)

df_prod_full = df_products.merge(df_depts, on='department_id').merge(df_aisles, on='aisle_id')
print(f"Catalog contains {len(df_products):,} products across {len(df_depts):,} departments and {len(df_aisles):,} aisles.")


---
## ❓ Business Question 1: What are the top 20 bestselling products and their reorder rates?
*Identifies core anchor items that drive customer loyalty and high reorder velocity.*


In [ ]:
prod_stats = df_op_sample.groupby('product_id').agg(
    total_purchases=('reordered', 'count'),
    total_reorders=('reordered', 'sum'),
    reorder_rate=('reordered', 'mean')
).reset_index().merge(df_prod_full, on='product_id')

top20 = prod_stats.sort_values(by='total_purchases', ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20, y='product_name', x='total_purchases', hue='reorder_rate', palette='viridis')
plt.title("Top 20 Most Purchased Products (Color = Reorder Rate)", fontsize=14, fontweight='bold')
plt.xlabel("Total Order Count (in sample)")
plt.ylabel("Product Name")
plt.show()


---
## ❓ Business Question 2: Which departments generate the highest volume and highest retention?
*Identifies revenue drivers vs. high-loyalty staple categories.*


In [ ]:
dept_stats = prod_stats.groupby('department').agg(
    total_volume=('total_purchases', 'sum'),
    mean_reorder_rate=('reorder_rate', 'mean')
).reset_index().sort_values(by='total_volume', ascending=False)

fig, ax1 = plt.subplots(figsize=(14, 5))

color = '#1f77b4'
ax1.set_xlabel('Department', fontweight='bold')
ax1.set_ylabel('Total Volume', color=color, fontweight='bold')
sns.barplot(data=dept_stats, x='department', y='total_volume', ax=ax1, color=color, alpha=0.8)
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('Reorder Rate', color=color, fontweight='bold')
sns.lineplot(data=dept_stats, x='department', y='mean_reorder_rate', ax=ax2, color=color, marker='o', linewidth=2.5)

plt.title("Department Volume vs. Department Reorder Rate", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## ❓ Business Question 3: How does add-to-cart position influence reorder probability?
*Top-of-mind staple items (milk, bananas) are added first into carts, exhibiting higher reorder probabilities.*


In [ ]:
cart_reorder = df_op_sample.groupby('add_to_cart_order')['reordered'].mean().reset_index()

plt.figure(figsize=(11, 4))
sns.lineplot(data=cart_reorder[cart_reorder['add_to_cart_order'] <= 25], x='add_to_cart_order', y='reordered', marker='o', color='#9b5de5', linewidth=2.5)
plt.title("Reorder Probability vs. Add-to-Cart Sequence Position", fontsize=13, fontweight='bold')
plt.xlabel("Add to Cart Order (Position in Basket)")
plt.ylabel("Reorder Probability")
plt.show()


---
## ❓ Business Question 4: Which product pairs co-occur most frequently in the same basket?
*Market basket analysis for recommendation cross-selling and knowledge graph relationships.*


In [ ]:
# High-frequency co-occurrence analysis on popular products
top_50_pids = set(prod_stats.sort_values(by='total_purchases', ascending=False).head(50)['product_id'])
df_top_op = df_op_sample[df_op_sample['product_id'].isin(top_50_pids)]

# Self-join on order_id to find pairs
order_pairs = df_top_op.merge(df_top_op, on='order_id')
order_pairs = order_pairs[order_pairs['product_id_x'] < order_pairs['product_id_y']]

pair_counts = order_pairs.groupby(['product_id_x', 'product_id_y']).size().reset_index(name='co_occurrence')
pair_counts = pair_counts.merge(df_products[['product_id', 'product_name']], left_on='product_id_x', right_on='product_id') \
                         .merge(df_products[['product_id', 'product_name']], left_on='product_id_y', right_on='product_id', suffixes=('_A', '_B'))

top_pairs = pair_counts.sort_values(by='co_occurrence', ascending=False).head(10)
top_pairs['pair_label'] = top_pairs['product_name_A'] + " + " + top_pairs['product_name_B']

plt.figure(figsize=(12, 5))
sns.barplot(data=top_pairs, y='pair_label', x='co_occurrence', color='#00b4d8')
plt.title("Top 10 Most Frequently Co-Purchased Product Pairs", fontsize=14, fontweight='bold')
plt.xlabel("Co-Occurrence Count in Basket Sample")
plt.ylabel("Product Pair")
plt.show()
